In [1]:
import pandas as pd
import numpy as np

# Display settings for cleaner browsing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [3]:
train.head()


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


### Check for missing values.

In [5]:
missing = train.isna().mean().sort_values(ascending=False)
missing



CryoSleep       0.024963
ShoppingMall    0.023927
VIP             0.023352
HomePlanet      0.023122
Name            0.023007
Cabin           0.022892
VRDeck          0.021627
FoodCourt       0.021051
Spa             0.021051
Destination     0.020936
RoomService     0.020821
Age             0.020591
PassengerId     0.000000
Transported     0.000000
dtype: float64

In [6]:
# Does missingness correlate with target?
missing_vs_target = (
    train.assign(Transported=train.Transported.astype(int))
         .isna()
         .groupby(train.Transported.astype(int))
         .mean()
         .T
         .sort_values(by=1, ascending=False)
)

missing_vs_target


Transported,0,1
ShoppingMall,0.021784,0.026039
CryoSleep,0.025724,0.024212
VIP,0.022943,0.023755
HomePlanet,0.022711,0.023527
Name,0.022943,0.023070
Cabin,0.022943,0.022841
FoodCourt,0.019467,0.022613
VRDeck,0.020857,0.022385
Destination,0.020857,0.021014
Spa,0.021321,0.020786


### Missingness is uniform across "Transported"

### Feature Extraction

In [7]:
def extract_cabin_features(df):
    # Split Cabin into Deck / Number / Side
    cabin_split = df["Cabin"].str.split("/", expand=True)
    df["Deck"] = cabin_split[0]
    df["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
    df["Side"] = cabin_split[2]
    return df


In [8]:
def extract_passenger_group_features(df):
    # GroupID is the part before the underscore
    df["GroupID"] = df["PassengerId"].str.split("_").str[0]

    # Compute group sizes (requires groupby)
    group_sizes = df.groupby("GroupID")["PassengerId"].transform("count")
    df["GroupSize"] = group_sizes

    # Solo traveler flag
    df["IsSolo"] = (df["GroupSize"] == 1).astype(int)
    return df


In [9]:
def extract_spending_features(df):
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

    # Total spend
    df["TotalSpend"] = df[spend_cols].sum(axis=1)

    # Zero spend flag
    df["ZeroSpend"] = (df["TotalSpend"] == 0).astype(int)

    # Log transform (avoid -inf)
    df["LogSpend"] = np.log1p(df["TotalSpend"])
    return df


In [10]:
for dataset in [train, test]:
    dataset = extract_cabin_features(dataset)
    dataset = extract_passenger_group_features(dataset)
    dataset = extract_spending_features(dataset)


### Check to see if Group ID corresponds with any cabin info.


In [11]:
# Extract group and cabin info
tmp = train[["GroupID", "Deck", "CabinNum", "Side"]].copy()

# Count unique values per group
group_cabin_consistency = (
    tmp.groupby("GroupID")
       .agg({
           "Deck": "nunique",
           "CabinNum": "nunique",
           "Side": "nunique"
       })
)

group_cabin_consistency.head()


,Deck,CabinNum,Side
GroupID,,,
0001,1,1,1
0002,1,1,1
0003,1,1,1
0004,1,1,1
0005,1,1,1


In [12]:
group_cabin_consistency.head(20)



,Deck,CabinNum,Side
GroupID,,,
0001,1,1,1
0002,1,1,1
0003,1,1,1
0004,1,1,1
0005,1,1,1
0006,2,2,1
0007,1,1,1
0008,1,1,1
0009,1,1,1


In [13]:
consistency = group_cabin_consistency

deck_consistency_rate = (consistency["Deck"] == 1).mean()
side_consistency_rate = (consistency["Side"] == 1).mean()
cabin_num_consistency_rate = (consistency["CabinNum"] == 1).mean()

deck_consistency_rate, side_consistency_rate, cabin_num_consistency_rate


(0.9163583722052436, 0.9840759208621521, 0.9165192214894644)

In [14]:
# Work on a copy to avoid modifying your original frame
df = train.copy()

# Extract GroupID if not already done
df["GroupID"] = df["PassengerId"].str.split("_").str[0]

# Only consider rows where Cabin is NOT missing
non_missing = df.dropna(subset=["Cabin"])

# Count unique cabin values per group
group_cabin_counts = (
    non_missing.groupby("GroupID")["Cabin"]
               .nunique()
)

# Groups with >1 unique cabin value
bad_groups = group_cabin_counts[group_cabin_counts > 1].index

# Show the actual conflicting cabin assignments
conflicting = (
    non_missing[non_missing["GroupID"].isin(bad_groups)]
        .sort_values(["GroupID", "Cabin"])
)

conflicting


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,GroupID,GroupSize,IsSolo,TotalSpend,ZeroSpend,LogSpend
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True,F,2.0,S,0006,2,0,1584.0,0,7.368340
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True,G,0.0,S,0006,2,0,0.0,1,0.000000
20,0017_02,Earth,False,F/6/P,55 Cancri e,14.0,False,412.0,0.0,1.0,0.0,679.0,Philda Brighttt,False,F,6.0,P,0017,2,0,1092.0,0,6.996681
19,0017_01,Earth,False,G/0/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Lyde Brighttt,True,G,0.0,P,0017,2,0,0.0,1,0.000000
59,0064_02,NaN,True,E/3/S,TRAPPIST-1e,33.0,False,0.0,0.0,NaN,0.0,0.0,Colatz Keen,True,E,3.0,S,0064,2,0,0.0,1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8655,9231_03,Mars,False,E/592/P,TRAPPIST-1e,22.0,False,0.0,0.0,0.0,0.0,0.0,Cus Paie,False,E,592.0,P,9231,3,0,0.0,1,0.000000
8653,9231_01,Mars,True,F/1888/P,55 Cancri e,2.0,False,0.0,0.0,0.0,0.0,0.0,Denuts Paie,True,F,1888.0,P,9231,3,0,0.0,1,0.000000
8654,9231_02,Mars,False,F/1888/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,0.0,0.0,Walls Paie,True,F,1888.0,P,9231,3,0,0.0,1,0.000000
8683,9272_02,Earth,False,F/1894/P,TRAPPIST-1e,21.0,False,86.0,3.0,149.0,208.0,329.0,Gordo Simson,False,F,1894.0,P,9272,2,0,775.0,0,6.654153


### Impute Cabin
If we see that there are multiple passengers in a group, those passengers share the same cabin over 90% of the time.

So for any blank cabin fields, we'll check to see if there are any other members of the same group that do have cabin filled out, and use a lookup to fill it in.

In [15]:
# Combine train + test for consistent group-level imputation
full = pd.concat([train, test], ignore_index=True)

# Compute unique counts per group
group_stats = (
    full.groupby("GroupID")
        .agg({
            "Deck": "nunique",
            "CabinNum": "nunique",
            "Side": "nunique"
        })
)

# Keep only groups with exactly 1 unique value for each cabin component
consistent_groups = group_stats[
    (group_stats["Deck"] == 1) &
    (group_stats["CabinNum"] == 1) &
    (group_stats["Side"] == 1)
].index

# For each consistent group, get the single cabin assignment
group_cabin_lookup = (
    full[full["GroupID"].isin(consistent_groups)]
        .groupby("GroupID")
        .agg({
            "Deck": "first",
            "CabinNum": "first",
            "Side": "first"
        })
)

def impute_cabin_from_group(df, lookup):
    df = df.copy()
    for col in ["Deck", "CabinNum", "Side"]:
        df[col] = df.apply(
            lambda row: lookup.loc[row["GroupID"], col]
            if pd.isna(row[col]) and row["GroupID"] in lookup.index
            else row[col],
            axis=1
        )
    return df

full_imputed = impute_cabin_from_group(full, group_cabin_lookup)

# Reconstruct Cabin ONLY where all components are non-missing
mask = (
    full_imputed["Deck"].notna() &
    full_imputed["CabinNum"].notna() &
    full_imputed["Side"].notna()
)

full_imputed.loc[mask, "Cabin"] = (
    full_imputed.loc[mask, "Deck"].astype(str) + "/" +
    full_imputed.loc[mask, "CabinNum"].astype(int).astype(str) + "/" +
    full_imputed.loc[mask, "Side"].astype(str)
)


train_imputed = full_imputed.iloc[:len(train)].copy()
test_imputed  = full_imputed.iloc[len(train):].copy()



In [16]:
train_imputed.head(20)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,GroupID,GroupSize,IsSolo,TotalSpend,ZeroSpend,LogSpend
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0.0,P,0001,1,1,0.0,1,0.000000
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0.0,S,0002,1,1,736.0,0,6.602588
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0.0,S,0003,2,0,10383.0,0,9.248021
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0.0,S,0003,2,0,5176.0,0,8.551981
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1.0,S,0004,1,1,1091.0,0,6.995766
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True,F,0.0,P,0005,1,1,774.0,0,6.652863
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True,F,2.0,S,0006,2,0,1584.0,0,7.368340
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True,G,0.0,S,0006,2,0,0.0,1,0.000000
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True,F,3.0,S,0007,1,1,1018.0,0,6.926577
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True,B,1.0,P,0008,3,0,0.0,1,0.000000


In [17]:
missing_mask = (
    full_imputed["Deck"].isna() |
    full_imputed["CabinNum"].isna() |
    full_imputed["Side"].isna()
)

missing_count = missing_mask.sum()
missing_count



181

### Impute the remaining missing cabins
 - Cabin # imputed based on group.  (Lower groups tend to have lower room numbers)
 - Deck and Side are distrubuted randomly based on real distrubution.

In [18]:
full_imputed["GroupID_num"] = full_imputed["GroupID"].astype(int)

# Real Side distribution
side_counts = full_imputed["Side"].value_counts(normalize=True, dropna=True)
p_side = side_counts.get("P", 0.5)
s_side = side_counts.get("S", 0.5)

# Real Deck distribution
deck_counts = full_imputed["Deck"].value_counts(normalize=True, dropna=True)
deck_values = deck_counts.index.tolist()
deck_probs = deck_counts.values.tolist()


def interpolate_cabin_by_group(df):
    df = df.copy()
    touched_idx = []

    for (deck, side), block in df.groupby(["Deck", "Side"], group_keys=False):
        known = block[block["CabinNum"].notna()]
        missing = block[block["CabinNum"].isna()]

        if len(known) < 2 or missing.empty:
            continue

        known = known.sort_values("GroupID_num")
        x = known["GroupID_num"].values
        y = known["CabinNum"].values

        m_idx = missing.index
        x_new = missing["GroupID_num"].values

        y_new = np.interp(x_new, x, y)

        df.loc[m_idx, "CabinNum"] = y_new
        touched_idx.extend(m_idx.tolist())

    return df, touched_idx

full_imputed, touched = interpolate_cabin_by_group(full_imputed)



mask = (
    full_imputed["Deck"].notna() &
    full_imputed["CabinNum"].notna() &
    full_imputed["Side"].notna()
)

full_imputed.loc[mask, "Cabin"] = (
    full_imputed.loc[mask, "Deck"].astype(str) + "/" +
    full_imputed.loc[mask, "CabinNum"].round().astype(int).astype(str) + "/" +
    full_imputed.loc[mask, "Side"].astype(str)
)

train_imputed = full_imputed.iloc[:len(train)].copy()
test_imputed  = full_imputed.iloc[len(train):].copy()

In [19]:
train_imputed.head(20)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,GroupID,GroupSize,IsSolo,TotalSpend,ZeroSpend,LogSpend,GroupID_num
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0.0,P,0001,1,1,0.0,1,0.000000,1
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0.0,S,0002,1,1,736.0,0,6.602588,2
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0.0,S,0003,2,0,10383.0,0,9.248021,3
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0.0,S,0003,2,0,5176.0,0,8.551981,3
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1.0,S,0004,1,1,1091.0,0,6.995766,4
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True,F,0.0,P,0005,1,1,774.0,0,6.652863,5
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True,F,2.0,S,0006,2,0,1584.0,0,7.368340,6
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True,G,0.0,S,0006,2,0,0.0,1,0.000000,6
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True,F,3.0,S,0007,1,1,1018.0,0,6.926577,7
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True,B,1.0,P,0008,3,0,0.0,1,0.000000,8


### Impute 0 spend for people in CryoSleep
Also, impute VIP = false... very few VIPs are in Cryo.

In [20]:
cols_to_zero = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Zero out spending for CryoSleep passengers
full_imputed.loc[full_imputed["CryoSleep"] == True, cols_to_zero] = 0.0

# VIP should be False for CryoSleep passengers
full_imputed.loc[full_imputed["CryoSleep"] == True, "VIP"] = False


### Impute 0 spend in 1 missing category, if all other categories are 0


In [21]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Count how many spending fields are zero
zero_count = (full_imputed[spend_cols] == 0).sum(axis=1)

# Identify rows where:
# - At least 4 spending fields are zero
# - AND at least one spending field is NaN
mask = (zero_count >= 4) & (full_imputed[spend_cols].isna().any(axis=1))

# Fill NaNs with 0 for those rows
full_imputed.loc[mask, spend_cols] = full_imputed.loc[mask, spend_cols].fillna(0)


### Impute Age, HomePlanet and Destination from Group if information exists, otherwise take ship average

In [22]:
global_age_median = full_imputed["Age"].median()

# Group median age
group_age_median = full_imputed.groupby("GroupID")["Age"].transform("median")

# Fill group ages first
full_imputed.loc[full_imputed["IsSolo"] == 0, "Age"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 0, "Age"]
    .fillna(group_age_median)
)

# Solo fallback
full_imputed.loc[full_imputed["IsSolo"] == 1, "Age"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 1, "Age"]
    .fillna(global_age_median)
)

global_homeplanet_mode = full_imputed["HomePlanet"].mode()[0]

group_homeplanet_mode = (
    full_imputed.groupby("GroupID")["HomePlanet"]
                .transform(lambda x: x.mode().iloc[0] if x.notna().any() else np.nan)
)

# Group first
full_imputed.loc[full_imputed["IsSolo"] == 0, "HomePlanet"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 0, "HomePlanet"]
    .fillna(group_homeplanet_mode)
)

# Solo fallback
full_imputed.loc[full_imputed["IsSolo"] == 1, "HomePlanet"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 1, "HomePlanet"]
    .fillna(global_homeplanet_mode)
)

global_dest_mode = full_imputed["Destination"].mode()[0]

group_dest_mode = (
    full_imputed.groupby("GroupID")["Destination"]
                .transform(lambda x: x.mode().iloc[0] if x.notna().any() else np.nan)
)

# Group first
full_imputed.loc[full_imputed["IsSolo"] == 0, "Destination"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 0, "Destination"]
    .fillna(group_dest_mode)
)

# Solo fallback
full_imputed.loc[full_imputed["IsSolo"] == 1, "Destination"] = (
    full_imputed.loc[full_imputed["IsSolo"] == 1, "Destination"]
    .fillna(global_dest_mode)
)



In [23]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# Count missing values in each spending column
full_imputed[spend_cols].isna().sum()


RoomService     159
FoodCourt       170
ShoppingMall    161
Spa             164
VRDeck          144
dtype: int64

In [24]:
full_imputed["SpendNaNCount"] = full_imputed[spend_cols].isna().sum(axis=1)

full_imputed.loc[full_imputed["SpendNaNCount"] > 0, 
                 ["PassengerId", "SpendNaNCount"] + spend_cols].head(20)


,PassengerId,SpendNaNCount,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
16,0014_01,1,1286.0,122.0,NaN,0.0,0.0
35,0031_03,1,NaN,0.0,1750.0,990.0,0.0
48,0050_01,1,790.0,0.0,0.0,NaN,0.0
95,0103_01,1,0.0,NaN,0.0,0.0,17.0
132,0141_01,1,NaN,0.0,97.0,0.0,0.0
143,0164_01,1,50.0,1688.0,0.0,NaN,135.0
170,0193_02,1,NaN,0.0,8.0,1072.0,0.0
181,0202_02,1,0.0,2433.0,NaN,878.0,443.0
199,0220_04,1,122.0,84.0,NaN,0.0,0.0
220,0234_01,1,NaN,888.0,0.0,0.0,0.0


### Impute remaining spend with median of category

In [25]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for col in spend_cols:
    median_val = full_imputed[col].median()
    full_imputed[col] = full_imputed[col].fillna(median_val)


In [26]:
train_imputed = full_imputed.iloc[:len(train)].copy()
test_imputed  = full_imputed.iloc[len(train):].copy()

In [27]:
train_imputed.head(20)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,CabinNum,Side,GroupID,GroupSize,IsSolo,TotalSpend,ZeroSpend,LogSpend,GroupID_num,SpendNaNCount
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0.0,P,0001,1,1,0.0,1,0.000000,1,0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0.0,S,0002,1,1,736.0,0,6.602588,2,0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0.0,S,0003,2,0,10383.0,0,9.248021,3,0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0.0,S,0003,2,0,5176.0,0,8.551981,3,0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1.0,S,0004,1,1,1091.0,0,6.995766,4,0
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True,F,0.0,P,0005,1,1,774.0,0,6.652863,5,0
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True,F,2.0,S,0006,2,0,1584.0,0,7.368340,6,0
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,0.0,Candra Jacostaffey,True,G,0.0,S,0006,2,0,0.0,1,0.000000,6,0
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True,F,3.0,S,0007,1,1,1018.0,0,6.926577,7,0
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True,B,1.0,P,0008,3,0,0.0,1,0.000000,8,0
